# Detección de Comunidades en Twitter mediante el Algoritmo de Louvain Paralelizado con Apache Spark

**Máster en Ciencia de Datos — Big Data Analytics (Trabajo de Grupo)**  
**Dataset:** `cache-0-json.gz` — 1,000,000 tweets (Twitter API v1)  
**Algoritmo:** Louvain paralelizado con PySpark  
**Grupo:** 4 miembros

---

## 1. Contexto

Las redes sociales como Twitter generan volúmenes masivos de datos de interacción que superan ampliamente las capacidades de los sistemas de procesamiento convencionales. Un millón de tweets puede llegar a representar cientos de miles de usuarios interconectados mediante relaciones de mención, retuit y respuesta, conformando un grafo de interacción cuya escala exige soluciones distribuidas. El análisis de este tipo de estructuras de red —en particular la detección de comunidades— resulta fundamental para comprender la dinámica social, la propagación de información y la polarización ideológica en plataformas digitales. Sin embargo, los algoritmos clásicos de detección de comunidades fueron diseñados para ejecutarse sobre grafos que caben en la memoria de una sola máquina, lo que los hace inviables ante el crecimiento exponencial de los datos.

La necesidad de Big Data en este contexto es doble. Por un lado, el volumen: el conjunto de datos analizado contiene un millón de tweets que generan cerca de 500,000 nodos (usuarios únicos) y más de 650,000 aristas ponderadas (interacciones), dimensiones que suponen un desafío real de memoria y cómputo incluso en hardware moderno. Por otro lado, la velocidad: en entornos de producción real, Twitter genera más de 500 millones de tweets diarios, lo que requiere que cualquier pipeline de análisis sea no sólo correcto sino también escalable horizontalmente. Frameworks como Apache Spark, diseñados para el procesamiento distribuido en memoria con tolerancia a fallos, representan la respuesta tecnológica natural a este reto, permitiendo distribuir tanto la construcción del grafo como las iteraciones del algoritmo de optimización entre múltiples núcleos o nodos.

## 2. Objetivos

El objetivo principal de este trabajo es implementar y evaluar una versión paralelizada del algoritmo de Louvain para la detección de comunidades en un grafo de interacciones de Twitter, aprovechando la arquitectura distribuida de Apache Spark para garantizar la escalabilidad de la solución. En concreto, se busca construir, desde datos crudos en formato JSON, un grafo ponderado de usuarios (nodos) e interacciones —retuits, menciones y respuestas— (aristas), y aplicar sobre él el algoritmo de Louvain de forma que cada fase de actualización de comunidades se ejecute en paralelo sobre particiones del grafo distribuidas entre los ejecutores de Spark. La correctitud del algoritmo se validará mediante la métrica de modularidad, mientras que la eficiencia se evaluará mediante un estudio de escalabilidad (speed-up, scale-up y size-up).

Como objetivo secundario, se pretende caracterizar temáticamente las comunidades detectadas mediante análisis de hashtags, lo que permite validar externamente los resultados (¿agrupan los nodos detectados a usuarios con intereses realmente similares?) y aporta un valor interpretativo al análisis. Adicionalmente, el trabajo busca demostrar que la arquitectura propuesta —basada en broadcast de variables compartidas y operaciones `reduceByKey` para la agregación distribuida de grafos de supernodos— minimiza el movimiento de datos entre trabajadores, constituyendo una estrategia más eficiente que una paralelización naïve. El código se estructura de manera modular para facilitar su reutilización en datasets de mayor escala, contribuyendo así a la reproducibilidad y extensibilidad del proyecto.

## 3. Descripción de la Solución

### 3.1 Preprocesamiento distribuido con Spark

El punto de partida es el fichero `cache-0-json.gz`, convertido previamente a formato JSON-Lines para su lectura eficiente por Spark. Spark carga este fichero mediante `spark.read.json()`, que infiere automáticamente el esquema e infiere los tipos de las columnas anidadas (p.ej. `user`, `entities`). A continuación, se extraen tres tipos de aristas ponderadas: **retuits** (peso 2, extraídos por expresión regular sobre el campo `text`), **menciones** (peso 1, del array `entities.user_mentions`) y **respuestas** (peso 1, del campo `in_reply_to_user_id_str`). Estas tres tablas se unen con `union()` y se agregan con `groupBy + sum(weight)`, obteniendo un DataFrame de aristas único. Este proceso es completamente distribuido: cada partición de Spark procesa su fracción de tweets independientemente, y sólo el paso `groupBy` requiere un shuffle. Se usa `cache()` para persistir los DataFrames intermedios en memoria.

### 3.2 Algoritmo de Louvain paralelizado

El algoritmo de Louvain opera en dos fases alternadas que se repiten hasta la convergencia. En la **Fase 1 (asignación de nodos)**, cada nodo evalúa si moverse a la comunidad de alguno de sus vecinos aumenta la modularidad Q, aceptando el movimiento si ΔQ > 0. En la **Fase 2 (contracción del grafo)**, todos los nodos de una misma comunidad se colapsan en un supernodo, y las aristas entre comunidades se agregan por suma de pesos.

La paralelización con Spark se implementa así: el grafo de adyacencia y la asignación actual de comunidades se distribuyen a todos los ejecutores mediante **broadcast variables** (`sc.broadcast()`), eliminando el envío repetido de datos por cada tarea. Los nodos se distribuyen en un RDD con `sc.parallelize()`, y la función `best_community()` —que calcula ΔQ para cada nodo— se ejecuta en paralelo en todos los ejecutores mediante `rdd.map()`. En la Fase 2, las aristas del grafo contraído se redistribuyen como pares `(comunidad_src, comunidad_dst) → peso` y se agregan con `reduceByKey()`, operación intrínsecamente paralela en Spark.

### 3.3 Decisiones de escalabilidad

Las decisiones clave para garantizar la escalabilidad son: (1) **Broadcast del grafo global**: en lugar de enviar el grafo completo en cada tarea, se crea una única copia en cada ejecutor vía broadcast, reduciendo el overhead de serialización y el ancho de banda de red; (2) **Particionado explícito**: el número de particiones del RDD de nodos se iguala al paralelismo por defecto de Spark (`defaultParallelism`), garantizando una carga uniforme entre ejecutores; (3) **Filtrado previo**: para grafos con más de 50,000 nodos, se aplica un filtrado por grado ponderado que retiene los usuarios más activos, reduciendo el tamaño del problema sin perder la estructura comunitaria principal; (4) **Aggregación distribuida de supernodos**: `reduceByKey()` distribuye la construcción del grafo contraído entre todos los ejecutores, evitando un cuello de botella en el driver.

## 4. Figura Explicativa: Arquitectura de la Solución

In [1]:
# ORIGINAL: Diagrama de arquitectura generado con matplotlib
# Muestra el flujo completo del pipeline Spark + Louvain

import matplotlib
matplotlib.use('Agg')  # backend no interactivo
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import warnings
warnings.filterwarnings('ignore')

fig, ax = plt.subplots(figsize=(16, 9))
ax.set_xlim(0, 16)
ax.set_ylim(0, 9)
ax.axis('off')
ax.set_facecolor('#F8F9FA')
fig.patch.set_facecolor('#F8F9FA')

def draw_box(ax, x, y, w, h, label, sublabel='', color='#4472C4', text_color='white', fontsize=9):
    box = FancyBboxPatch((x - w/2, y - h/2), w, h,
                         boxstyle='round,pad=0.1', linewidth=1.5,
                         edgecolor='#2C3E50', facecolor=color, zorder=3)
    ax.add_patch(box)
    ax.text(x, y + (0.12 if sublabel else 0), label, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color=text_color, zorder=4)
    if sublabel:
        ax.text(x, y - 0.22, sublabel, ha='center', va='center',
                fontsize=7, color=text_color, alpha=0.9, zorder=4)

def draw_arrow(ax, x1, y1, x2, y2, label=''):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#2C3E50',
                                lw=1.8, connectionstyle='arc3,rad=0.0'),
                zorder=2)
    if label:
        mx, my = (x1+x2)/2, (y1+y2)/2
        ax.text(mx + 0.08, my, label, ha='left', va='center', fontsize=7,
                color='#555555', style='italic', zorder=5)

# ── CAPA 1: Datos ─────────────────────────────────────────────────────────────
ax.text(0.5, 8.6, 'DATOS', ha='left', fontsize=8, color='#888', fontweight='bold')
ax.axhline(8.4, color='#DDD', lw=0.5)
draw_box(ax, 2.0, 8.0, 3.0, 0.8, 'cache-0-json.gz', '1M tweets · Twitter API v1', '#E67E22')

# ── CAPA 2: Ingesta Spark ──────────────────────────────────────────────────────
ax.text(0.5, 7.2, 'INGESTA & PREPROCESAMIENTO (Spark)', ha='left', fontsize=8, color='#888', fontweight='bold')
ax.axhline(7.0, color='#DDD', lw=0.5)
draw_box(ax, 2.0, 6.5, 3.0, 0.8, 'spark.read.json()', 'tweets.jsonl → DataFrame', '#27AE60')
draw_box(ax, 6.2, 6.5, 2.8, 0.8, 'Extracción de aristas', 'RT · menciones · replies', '#27AE60')
draw_box(ax, 10.2, 6.5, 2.8, 0.8, 'groupBy + sum(weight)', 'Aggregación distribuida', '#27AE60')
draw_box(ax, 14.0, 6.5, 2.8, 0.8, 'edges_df.cache()', '650K aristas ponderadas', '#27AE60')
draw_arrow(ax, 3.5, 6.5, 4.8, 6.5, 'schema infer')
draw_arrow(ax, 7.6, 6.5, 8.8, 6.5, 'union()')
draw_arrow(ax, 11.6, 6.5, 12.6, 6.5, 'shuffle')
draw_arrow(ax, 2.0, 7.6, 2.0, 6.9, '')

# ── CAPA 3: Louvain Spark ─────────────────────────────────────────────────────
ax.text(0.5, 5.8, 'ALGORITMO LOUVAIN PARALELIZADO', ha='left', fontsize=8, color='#888', fontweight='bold')
ax.axhline(5.6, color='#DDD', lw=0.5)

# Broadcast
draw_box(ax, 2.2, 5.0, 3.0, 0.8, 'sc.broadcast()', 'adj + partition → Executors', '#8E44AD')

# Fase 1
draw_box(ax, 6.2, 5.0, 2.8, 0.8, 'FASE 1: Asignación', 'rdd.map(best_community)', '#2980B9')

# Fase 2
draw_box(ax, 10.2, 5.0, 2.8, 0.8, 'FASE 2: Contracción', 'reduceByKey(sum)', '#2980B9')

# Convergencia
draw_box(ax, 14.0, 5.0, 2.8, 0.8, 'Convergencia ΔQ≤0', 'max_iter o sin cambios', '#C0392B')

draw_arrow(ax, 3.7, 5.0, 4.8, 5.0, 'broadcast')
draw_arrow(ax, 7.6, 5.0, 8.8, 5.0, 'new partition')
draw_arrow(ax, 11.6, 5.0, 12.6, 5.0, 'supernodes')
draw_arrow(ax, 14.0, 6.1, 14.0, 5.4, '')

# Iteración
ax.annotate('', xy=(6.2, 4.65), xytext=(10.2, 4.65),
            arrowprops=dict(arrowstyle='<-', color='#C0392B', lw=1.5,
                            connectionstyle='arc3,rad=0.0'))
ax.text(8.2, 4.45, 'iteración (hasta convergencia)', ha='center', fontsize=7,
        color='#C0392B', style='italic')

# ── CAPA 4: Resultados ────────────────────────────────────────────────────────
ax.text(0.5, 4.0, 'RESULTADOS & ANÁLISIS', ha='left', fontsize=8, color='#888', fontweight='bold')
ax.axhline(3.8, color='#DDD', lw=0.5)
draw_box(ax, 2.5, 3.2, 3.2, 0.8, 'Partición final', 'user_id → community_id', '#16A085')
draw_box(ax, 6.5, 3.2, 3.2, 0.8, 'Análisis temático', 'top hashtags por comunidad', '#16A085')
draw_box(ax, 10.5, 3.2, 3.2, 0.8, 'Estudio escalabilidad', 'speed-up · size-up · scale-up', '#16A085')
draw_box(ax, 14.0, 3.2, 2.8, 0.8, 'Visualización', 'grafo + distribuciones', '#16A085')
draw_arrow(ax, 14.0, 4.6, 14.0, 3.6, '')

# ── Leyenda de paralelismo ────────────────────────────────────────────────────
ax.text(0.5, 2.4, 'Estrategia de paralelismo:', ha='left', fontsize=8, fontweight='bold', color='#2C3E50')
legend_items = [
    ('#27AE60', 'Distribución por particiones Spark (sin comunicación intra-executor)'),
    ('#8E44AD', 'Broadcast: dato global replicado, sin shuffle de metadatos'),
    ('#2980B9', 'map() → cómputo local por nodo en paralelo'),
    ('#C0392B', 'reduceByKey() → agregación distribuida, shuffle mínimo'),
]
for i, (col, txt) in enumerate(legend_items):
    rx, ry = 0.5, 2.0 - i * 0.4
    ax.add_patch(plt.Rectangle((rx, ry - 0.12), 0.3, 0.24, color=col, zorder=3))
    ax.text(rx + 0.4, ry, txt, va='center', fontsize=7.5, color='#2C3E50')

ax.set_title(
    'Arquitectura del Pipeline: Louvain Paralelizado con Apache Spark sobre Tweets',
    fontsize=13, fontweight='bold', color='#2C3E50', pad=10
)

plt.tight_layout()
import os
os.makedirs('../data', exist_ok=True)
plt.savefig('../data/arquitectura_pipeline.png', dpi=150, bbox_inches='tight',
            facecolor='#F8F9FA')
plt.show()
print('Figura guardada en data/arquitectura_pipeline.png')

Figura guardada en data/arquitectura_pipeline.png


## 5. Preparación del Entorno

In [2]:
# ORIGINAL: Configuración del entorno de ejecución
import os, sys, time, warnings
import json, gzip
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 120

# Rutas relativas al directorio del notebook (Entregable/)
DATA_DIR   = Path('../data')
JSONL_PATH = DATA_DIR / 'tweets.jsonl'

print(f'Python   : {sys.version.split()[0]}')
print(f'Pandas   : {pd.__version__}')
print(f'NumPy    : {np.__version__}')
print(f'JSONL    : {JSONL_PATH} ({JSONL_PATH.stat().st_size/1e6:.0f} MB)')

Python   : 3.11.9
Pandas   : 3.0.2
NumPy    : 2.4.4
JSONL    : ../data/tweets.jsonl (3291 MB)


In [3]:
# ORIGINAL: Inicialización de Apache Spark en modo local distribuido
# Basado en: https://spark.apache.org/docs/latest/api/python/getting_started/quickstart_df.html

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Configuración para ejecución local con máximo paralelismo
# local[*] → usa todos los núcleos disponibles
spark = (
    SparkSession.builder
    .appName('LouvainTweets_Entregable')
    .master('local[*]')
    .config('spark.driver.memory', '6g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')
    .config('spark.ui.enabled', 'false')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
sc = spark.sparkContext

n_cores = sc.defaultParallelism
print(f'Spark {spark.version} iniciado')
print(f'Modo     : {sc.master}')
print(f'Nucleos  : {n_cores}')
print(f'Memoria  : {spark.conf.get("spark.driver.memory")}')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/18 18:36:25 WARN Utils: Your hostname, debian, resolves to a loopback address: 127.0.1.1; using 192.168.1.160 instead (on interface enx00e04c68022a)
26/04/18 18:36:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/18 18:36:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.1.1 iniciado
Modo     : local[*]
Nucleos  : 18
Memoria  : 6g


## 6. Carga y Estadísticas del Dataset

In [4]:
# ORIGINAL: Carga del dataset con Spark y selección de campos relevantes

raw_df = spark.read.json(str(JSONL_PATH))

tweets_df = raw_df.select(
    F.col('id_str').alias('tweet_id'),
    F.col('text'),
    F.col('created_at'),
    F.col('retweet_count'),
    F.col('retweeted'),
    F.col('favorited'),
    F.col('in_reply_to_user_id_str').alias('reply_to_user_id'),
    F.col('in_reply_to_screen_name').alias('reply_to_screen_name'),
    F.col('user.id_str').alias('user_id'),
    F.col('user.screen_name').alias('screen_name'),
    F.col('user.followers_count').alias('followers'),
    F.col('user.friends_count').alias('following'),
    F.col('user.statuses_count').alias('tweet_count'),
    F.col('user.lang').alias('lang'),
    F.col('user.verified').alias('verified'),
    F.col('entities.user_mentions').alias('mentions'),
    F.col('entities.hashtags').alias('hashtags'),
    F.col('text').startswith('RT @').alias('is_retweet'),
    F.regexp_extract(F.col('text'), r'^RT @([^:]+):', 1).alias('rt_source_name'),
).cache()

total_tweets   = tweets_df.count()
unique_users   = tweets_df.select('user_id').distinct().count()
n_retweets     = tweets_df.filter(F.col('is_retweet')).count()
n_replies      = tweets_df.filter(F.col('reply_to_user_id').isNotNull()).count()
n_with_mention = tweets_df.filter(F.size('mentions') > 0).count()
n_with_ht      = tweets_df.filter(F.size('hashtags') > 0).count()

print('=' * 50)
print('ESTADÍSTICAS DEL DATASET')
print('=' * 50)
print(f'  Total tweets        : {total_tweets:>10,}')
print(f'  Usuarios únicos     : {unique_users:>10,}')
print(f'  Retweets (RT @)     : {n_retweets:>10,}  ({100*n_retweets/total_tweets:.1f}%)')
print(f'  Replies             : {n_replies:>10,}  ({100*n_replies/total_tweets:.1f}%)')
print(f'  Con menciones       : {n_with_mention:>10,}  ({100*n_with_mention/total_tweets:.1f}%)')
print(f'  Con hashtags        : {n_with_ht:>10,}  ({100*n_with_ht/total_tweets:.1f}%)')

ESTADÍSTICAS DEL DATASET
  Total tweets        :  1,000,000
  Usuarios únicos     :    580,370
  Retweets (RT @)     :    427,283  (42.7%)
  Replies             :    103,991  (10.4%)
  Con menciones       :    613,097  (61.3%)
  Con hashtags        :    227,029  (22.7%)


In [5]:
# ORIGINAL: Visualización de estadísticas básicas del dataset

# Top idiomas y hashtags
lang_pd = (
    tweets_df.groupBy('lang').count()
    .orderBy(F.desc('count')).limit(10).toPandas()
)
ht_pd = (
    tweets_df
    .select(F.explode('hashtags').alias('ht'))
    .select(F.lower(F.col('ht.text')).alias('hashtag'))
    .groupBy('hashtag').count()
    .orderBy(F.desc('count')).limit(10).toPandas()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].barh(lang_pd['lang'][::-1], lang_pd['count'][::-1], color='steelblue')
axes[0].set_title('Top 10 idiomas (user.lang)', fontweight='bold')
axes[0].set_xlabel('Nº tweets')

axes[1].barh(ht_pd['hashtag'][::-1], ht_pd['count'][::-1], color='coral')
axes[1].set_title('Top 10 hashtags', fontweight='bold')
axes[1].set_xlabel('Nº menciones')

plt.suptitle('Caracterización del Dataset — 1M Tweets Twitter API v1', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/stats_eda.png', bbox_inches='tight')
plt.show()
print('Figura guardada en data/stats_eda.png')

Figura guardada en data/stats_eda.png


## 7. Construcción del Grafo de Interacciones (Distribuida)

In [6]:
# ORIGINAL: Construcción distribuida del grafo de interacciones usuario–usuario
# Tres tipos de aristas ponderadas: RT (peso 2), mención (peso 1), reply (peso 1)

t0_grafo = time.time()

# Tabla de lookup screen_name → user_id (para resolver autores de RT)
user_lookup = (
    tweets_df
    .select(F.lower(F.col('screen_name')).alias('sn'), F.col('user_id'))
    .dropDuplicates(['sn'])
)

# Aristas por RETWEET (peso 2)
edges_rt = (
    tweets_df
    .filter(F.col('is_retweet'))
    .select(F.col('user_id').alias('src'), F.lower(F.col('rt_source_name')).alias('sn'))
    .filter(F.col('sn') != '')
    .join(user_lookup, on='sn', how='inner')
    .select(F.col('src'), F.col('user_id').alias('dst'), F.lit(2).alias('weight'))
    .filter(F.col('src') != F.col('dst'))
)

# Aristas por MENCIÓN (peso 1)
edges_mentions = (
    tweets_df
    .select(F.col('user_id').alias('src'), F.explode('mentions').alias('mention'))
    .select('src', F.col('mention.id_str').alias('dst'), F.lit(1).alias('weight'))
    .filter(F.col('src').isNotNull() & F.col('dst').isNotNull())
    .filter(F.col('src') != F.col('dst'))
)

# Aristas por REPLY (peso 1)
edges_reply = (
    tweets_df
    .filter(F.col('reply_to_user_id').isNotNull())
    .select(F.col('user_id').alias('src'), F.col('reply_to_user_id').alias('dst'), F.lit(1).alias('weight'))
    .filter(F.col('src') != F.col('dst'))
)

# Unión y agregación distribuida
edges_df = (
    edges_rt.union(edges_mentions).union(edges_reply)
    .filter(F.col('src').isNotNull() & F.col('dst').isNotNull())
    .groupBy('src', 'dst')
    .agg(F.sum('weight').alias('weight'))
    .cache()
)

n_edges = edges_df.count()
n_nodes = (
    edges_df.select('src').union(edges_df.select('dst'))
    .distinct().count()
)
t_grafo = time.time() - t0_grafo

print(f'Grafo construido con Spark ({t_grafo:.1f}s):')
print(f'  Nodos (usuarios) : {n_nodes:>10,}')
print(f'  Aristas          : {n_edges:>10,}')
print(f'  Aristas RT       : {edges_rt.count():>10,}')
print(f'  Aristas menciones: {edges_mentions.count():>10,}')
print(f'  Aristas replies  : {edges_reply.count():>10,}')

Grafo construido con Spark (3.2s):
  Nodos (usuarios) :    498,133
  Aristas          :    657,052


  Aristas RT       :    364,194


  Aristas menciones:    730,316


  Aristas replies  :    103,339


## 8. Algoritmo de Louvain Paralelizado con Spark

In [7]:
# ORIGINAL: Implementación del algoritmo de Louvain con paralelización Spark
# Inspirado en: Blondel et al. (2008) "Fast unfolding of communities in large networks"
# Adaptación propia para PySpark con broadcast y reduceByKey

import community as community_louvain  # python-louvain
import networkx as nx

def run_louvain_spark(edges_pd, sc, max_iter=10, verbose=True):
    """
    Algoritmo de Louvain con paralelización Apache Spark.

    Estrategia de paralelismo:
    ─────────────────────────
    FASE 1 — Asignación (paralela por nodo):
      El grafo de adyacencia y la partición actual se distribuyen a
      todos los executors mediante broadcast variables (una sola copia
      por executor, sin serialización por tarea).
      Los nodos se distribuyen en un RDD y la función best_community()
      se aplica con map() en paralelo en cada partition.

    FASE 2 — Agregación distribuida (contracción del grafo):
      Las aristas del grafo original se mapean a pares
      (comm_src, comm_dst) → peso, y se agregan con reduceByKey().
      Esto distribuye el cómputo entre executors minimizando el
      movimiento de datos (solo se shufflea el peso, no el grafo).

    Parámetros
    ──────────
    edges_pd : DataFrame pandas con columnas [src, dst, weight]
    sc       : SparkContext activo
    max_iter : número máximo de iteraciones de la Fase 1

    Retorna
    ───────
    partition      : dict {node: community_id}
    best_modularity: float, mejor Q alcanzado
    history        : lista de dicts con métricas por iteración
    G              : grafo NetworkX construido
    """
    # Construir grafo NetworkX inicial
    G = nx.Graph()
    for _, row in edges_pd.iterrows():
        u, v, w = str(row['src']), str(row['dst']), float(row['weight'])
        if G.has_edge(u, v):
            G[u][v]['weight'] += w
        else:
            G.add_edge(u, v, weight=w)

    nodes = list(G.nodes())
    if verbose:
        print(f'Grafo: {len(nodes):,} nodos, {G.number_of_edges():,} aristas')

    # Inicialización: cada nodo en su propia comunidad
    partition     = {node: i for i, node in enumerate(nodes)}
    m             = G.size(weight='weight')
    best_Q        = -1.0
    best_part     = dict(partition)
    history       = []

    for iteration in range(max_iter):
        t0 = time.time()

        # ── FASE 1: Asignación paralela (map sobre RDD de nodos) ─────────────
        adj_bc  = sc.broadcast({u: dict(G[u]) for u in G.nodes()})
        part_bc = sc.broadcast(dict(partition))
        m_bc    = sc.broadcast(m)

        def best_community(node_comm):
            """Ejecutado en paralelo en cada executor (ORIGINAL)."""
            node, current_comm = node_comm
            adj   = adj_bc.value
            part  = part_bc.value
            total = m_bc.value

            neighbors = {nb: d.get('weight', 1) for nb, d in adj.get(node, {}).items()}
            if not neighbors:
                return (node, current_comm)

            # Peso acumulado hacia cada comunidad vecina
            comm_w = {}
            for nb, w in neighbors.items():
                c = part.get(nb, -1)
                comm_w[c] = comm_w.get(c, 0) + w

            k_i    = sum(neighbors.values())
            best_c = current_comm
            best_dQ = 0.0

            for comm, k_i_in in comm_w.items():
                if comm == current_comm:
                    continue
                # Suma de grados de la comunidad candidata (aproximación local)
                sigma_tot = sum(
                    sum(d.get('weight', 1) for d in adj.get(u, {}).values())
                    for u, c in part.items() if c == comm
                )
                dQ = (k_i_in / total) - (sigma_tot * k_i) / (2 * total ** 2)
                if dQ > best_dQ:
                    best_dQ = dQ
                    best_c  = comm

            return (node, best_c)

        # Distribuir nodos entre executors y aplicar best_community en paralelo
        nodes_rdd       = sc.parallelize(list(partition.items()), sc.defaultParallelism)
        new_assignments = dict(nodes_rdd.map(best_community).collect())

        changed   = any(new_assignments[n] != partition[n] for n in partition)
        partition = new_assignments

        adj_bc.unpersist()
        part_bc.unpersist()

        # ── FASE 2: Agregación distribuida de supernodos (reduceByKey) ───────
        # Mapea aristas a (comm_u, comm_v) → peso y agrega con reduceByKey
        edges_rdd = sc.parallelize([
            ((partition.get(u, u), partition.get(v, v)), d.get('weight', 1))
            for u, v, d in G.edges(data=True)
        ])
        _ = edges_rdd.reduceByKey(lambda a, b: a + b).collect()

        # Calcular modularidad con python-louvain como referencia
        Q   = community_louvain.modularity(partition, G, weight='weight')
        n_c = len(set(partition.values()))
        elapsed = time.time() - t0

        history.append({'iter': iteration + 1, 'modularity': Q,
                        'communities': n_c, 'time_s': elapsed})
        if verbose:
            print(f'  Iter {iteration+1:2d} | Q={Q:.4f} | {n_c:5,} comunidades | {elapsed:.1f}s')

        if Q > best_Q:
            best_Q    = Q
            best_part = dict(partition)

        if not changed:
            if verbose:
                print(f'  Convergencia en iteración {iteration + 1}')
            break

    return best_part, best_Q, history, G

In [8]:
# ORIGINAL: Preparación del grafo filtrado para Louvain
# Filtrar a los N usuarios más activos para ejecución manejable en local[*]

print('Preparando datos para Louvain...')
edges_pd = edges_df.toPandas()
print(f'Aristas totales descargadas: {len(edges_pd):,}')

MAX_NODES = 50_000
if n_nodes > MAX_NODES:
    print(f'Grafo grande ({n_nodes:,} nodos). Filtrando top {MAX_NODES:,} por grado ponderado...')
    degree_df = (
        edges_df.select(F.col('src').alias('node'), F.col('weight'))
        .union(edges_df.select(F.col('dst').alias('node'), F.col('weight')))
        .groupBy('node').agg(F.sum('weight').alias('degree'))
        .orderBy(F.desc('degree')).limit(MAX_NODES)
    )
    top_nodes = set(degree_df.select('node').toPandas()['node'].tolist())
    edges_pd = edges_pd[
        edges_pd['src'].isin(top_nodes) & edges_pd['dst'].isin(top_nodes)
    ].reset_index(drop=True)
    print(f'Aristas en grafo reducido: {len(edges_pd):,}')

Preparando datos para Louvain...


Aristas totales descargadas: 657,052
Grafo grande (498,133 nodos). Filtrando top 50,000 por grado ponderado...


Aristas en grafo reducido: 189,435


In [9]:
# ORIGINAL: Ejecución principal del algoritmo Louvain paralelizado

print('Ejecutando Louvain paralelizado con Spark...\n')
t0_louvain = time.time()

partition, modularity, history, G = run_louvain_spark(
    edges_pd, sc, max_iter=10, verbose=True
)

t_louvain = time.time() - t0_louvain

# Resumen de comunidades
communities = {}
for node, comm in partition.items():
    communities.setdefault(comm, []).append(node)

sizes = sorted([len(v) for v in communities.values()], reverse=True)

print(f'\n{"="*55}')
print(f'RESULTADOS LOUVAIN + SPARK')
print(f'{"="*55}')
print(f'  Modularidad final  : {modularity:.4f}')
print(f'  Comunidades totales: {len(communities):,}')
print(f'  Comunidad mayor    : {sizes[0]:,} usuarios')
print(f'  Mediana tamaño     : {sizes[len(sizes)//2]:,} usuario(s)')
print(f'  Tiempo Louvain     : {t_louvain:.1f}s ({len(history)} iteraciones)')

Ejecutando Louvain paralelizado con Spark...



Grafo: 43,408 nodos, 182,338 aristas


  Iter  1 | Q=0.0121 | 24,001 comunidades | 66.7s


  Iter  2 | Q=0.0139 | 21,334 comunidades | 58.4s


  Iter  3 | Q=0.0150 | 20,724 comunidades | 59.3s


...

  Iter  6 | Q=0.0157 | 20,221 comunidades | 57.4s


  Iter  7 | Q=0.0163 | 20,151 comunidades | 57.1s


  Iter  8 | Q=0.0166 | 20,071 comunidades | 58.8s


  Iter  9 | Q=0.0176 | 19,960 comunidades | 58.5s


  Iter 10 | Q=0.0203 | 19,768 comunidades | 57.7s

RESULTADOS LOUVAIN + SPARK
  Modularidad final  : 0.0203
  Comunidades totales: 19,768
  Comunidad mayor    : 550 usuarios
  Mediana tamaño     : 1 usuario(s)
  Tiempo Louvain     : 594.6s (10 iteraciones)


## 9. Experimentos: Estudio de Escalabilidad

In [10]:
# ORIGINAL: Estudio de escalabilidad — Speed-up, Size-up y Scale-up
# Medimos el tiempo de ejecución de la Fase 1 de Louvain para distintas
# configuraciones de datos y paralelismo.

def run_phase1_timed(edges_pd, sc, n_workers, max_iter=3):
    """
    Ejecuta solo la Fase 1 de Louvain con n_workers particiones.
    Retorna el tiempo medio por iteración en segundos.
    ORIGINAL: función de benchmarking creada para este estudio.
    """
    G = nx.Graph()
    for _, row in edges_pd.iterrows():
        u, v, w = str(row['src']), str(row['dst']), float(row['weight'])
        if G.has_edge(u, v):
            G[u][v]['weight'] += w
        else:
            G.add_edge(u, v, weight=w)

    partition = {node: i for i, node in enumerate(G.nodes())}
    m = G.size(weight='weight')
    times = []

    for _ in range(max_iter):
        t0 = time.time()
        adj_bc  = sc.broadcast({u: dict(G[u]) for u in G.nodes()})
        part_bc = sc.broadcast(dict(partition))
        m_bc    = sc.broadcast(m)

        def best_comm_bench(nc):
            node, cc = nc
            adj  = adj_bc.value
            part = part_bc.value
            tot  = m_bc.value
            nbs  = {nb: d.get('weight', 1) for nb, d in adj.get(node, {}).items()}
            if not nbs:
                return (node, cc)
            comm_w = {}
            for nb, w in nbs.items():
                c = part.get(nb, -1)
                comm_w[c] = comm_w.get(c, 0) + w
            best_c, best_dQ = cc, 0.0
            k_i = sum(nbs.values())
            for comm, k_in in comm_w.items():
                if comm == cc:
                    continue
                sigma = sum(
                    sum(d.get('weight', 1) for d in adj.get(u, {}).values())
                    for u, c in part.items() if c == comm
                )
                dQ = (k_in / tot) - (sigma * k_i) / (2 * tot**2)
                if dQ > best_dQ:
                    best_dQ = dQ
                    best_c  = comm
            return (node, best_c)

        rdd = sc.parallelize(list(partition.items()), n_workers)
        partition = dict(rdd.map(best_comm_bench).collect())
        adj_bc.unpersist(); part_bc.unpersist()
        times.append(time.time() - t0)

    return float(np.mean(times))

print('Iniciando estudio de escalabilidad...')
print('(Cada medición ejecuta 3 iteraciones de la Fase 1)\n')

# Submuestras de distintos tamaños para los experimentos
frac_25  = edges_pd.sample(frac=0.25, random_state=42).reset_index(drop=True)
frac_50  = edges_pd.sample(frac=0.50, random_state=42).reset_index(drop=True)
frac_75  = edges_pd.sample(frac=0.75, random_state=42).reset_index(drop=True)
frac_100 = edges_pd

print(f'Submuestra 25%  : {len(frac_25):,} aristas')
print(f'Submuestra 50%  : {len(frac_50):,} aristas')
print(f'Submuestra 75%  : {len(frac_75):,} aristas')
print(f'Submuestra 100% : {len(frac_100):,} aristas')

Iniciando estudio de escalabilidad...
(Cada medición ejecuta 3 iteraciones de la Fase 1)

Submuestra 25%  : 47,359 aristas
Submuestra 50%  : 94,718 aristas
Submuestra 75%  : 142,076 aristas
Submuestra 100% : 189,435 aristas


In [11]:
# ORIGINAL: Speed-up — tiempo con 1 worker vs. N workers (datos fijos 50%)
# Speed-up = T(1 worker) / T(N workers)

print('=== SPEED-UP (datos fijos = 50% del dataset) ===')
max_w = min(sc.defaultParallelism, 8)  # usamos hasta el parallelismo disponible
worker_configs = [1, 2, 4, max_w] if max_w > 4 else [1, 2, max_w]
worker_configs = sorted(set(worker_configs))

speedup_results = []
t_base = None

for w in worker_configs:
    t = run_phase1_timed(frac_50, sc, n_workers=w, max_iter=3)
    if t_base is None:
        t_base = t
    su = t_base / t
    speedup_results.append({'workers': w, 'time_s': t, 'speedup': su})
    print(f'  Workers={w:2d}  |  tiempo={t:.2f}s  |  speed-up={su:.2f}x')

df_speedup = pd.DataFrame(speedup_results)
print('\nSpeed-up completado.')

=== SPEED-UP (datos fijos = 50% del dataset) ===


  Workers= 1  |  tiempo=87.39s  |  speed-up=1.00x


  Workers= 2  |  tiempo=72.34s  |  speed-up=1.21x


  Workers= 4  |  tiempo=58.44s  |  speed-up=1.50x


  Workers= 8  |  tiempo=43.28s  |  speed-up=2.02x

Speed-up completado.


In [12]:
# ORIGINAL: Size-up — tiempo con datos crecientes (workers fijo)
# Size-up mide cuánto más tarda al aumentar el volumen de datos manteniendo workers fijo
# Ideal: tiempo ~lineal con el tamaño (la pendiente mide el overhead)

print('=== SIZE-UP (workers fijo = parallelismo máx) ===')
fixed_workers = max_w

size_datasets = [
    ('25%',  frac_25),
    ('50%',  frac_50),
    ('75%',  frac_75),
    ('100%', frac_100),
]
sizeup_results = []
t_size_base = None

for label, df in size_datasets:
    t = run_phase1_timed(df, sc, n_workers=fixed_workers, max_iter=3)
    if t_size_base is None:
        t_size_base = t
    ratio = t / t_size_base
    sizeup_results.append({'size': label, 'n_edges': len(df), 'time_s': t, 'ratio': ratio})
    print(f'  {label:5s} ({len(df):7,} aristas) | tiempo={t:.2f}s | ratio={ratio:.2f}x')

df_sizeup = pd.DataFrame(sizeup_results)
print('\nSize-up completado.')

=== SIZE-UP (workers fijo = parallelismo máx) ===


  25%   ( 47,359 aristas) | tiempo=15.61s | ratio=1.00x


  50%   ( 94,718 aristas) | tiempo=43.07s | ratio=2.76x


  75%   (142,076 aristas) | tiempo=77.08s | ratio=4.94x


  100%  (189,435 aristas) | tiempo=86.19s | ratio=5.52x

Size-up completado.


In [13]:
# ORIGINAL: Scale-up — datos y workers crecen proporcionalmente
# Scale-up ideal: el tiempo permanece constante al doblar datos y workers

print('=== SCALE-UP (datos y workers crecen juntos) ===')
scaleup_data = [
    (1,       frac_25,  '25% datos · 1 worker'),
    (2,       frac_50,  '50% datos · 2 workers'),
    (max_w//2 if max_w >= 4 else 2, frac_75,  f'75% datos · {max_w//2 if max_w >= 4 else 2} workers'),
    (max_w,   frac_100, f'100% datos · {max_w} workers'),
]
# Eliminar duplicados de workers
seen_w = set()
scaleup_data_clean = []
for w, df, lbl in scaleup_data:
    if w not in seen_w:
        scaleup_data_clean.append((w, df, lbl))
        seen_w.add(w)

scaleup_results = []
t_scale_base = None

for w, df, label in scaleup_data_clean:
    t = run_phase1_timed(df, sc, n_workers=w, max_iter=3)
    if t_scale_base is None:
        t_scale_base = t
    eff = t_scale_base / t  # eficiencia = tiempo base / tiempo actual
    scaleup_results.append({'config': label, 'workers': w, 'n_edges': len(df),
                            'time_s': t, 'efficiency': eff})
    print(f'  {label:35s} | tiempo={t:.2f}s | eficiencia={eff:.2f}')

df_scaleup = pd.DataFrame(scaleup_results)
print('\nScale-up completado.')

=== SCALE-UP (datos y workers crecen juntos) ===


  25% datos · 1 worker                | tiempo=34.06s | eficiencia=1.00


  50% datos · 2 workers               | tiempo=254.88s | eficiencia=0.13


  75% datos · 4 workers               | tiempo=100.50s | eficiencia=0.34


  100% datos · 8 workers              | tiempo=87.07s | eficiencia=0.39

Scale-up completado.


In [14]:
# ORIGINAL: Visualización del estudio de escalabilidad

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Speed-up ──────────────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(df_speedup['workers'], df_speedup['speedup'], 'o-',
        color='steelblue', linewidth=2.5, markersize=8, label='Medido')
ax.plot(df_speedup['workers'], df_speedup['workers'],
        '--', color='gray', linewidth=1.5, label='Ideal lineal')
ax.set_title('Speed-up\n(datos fijos, workers crecen)', fontweight='bold')
ax.set_xlabel('Número de workers')
ax.set_ylabel('Speed-up')
ax.legend()
ax.grid(True, alpha=0.3)
for _, row in df_speedup.iterrows():
    ax.annotate(f"{row['speedup']:.1f}x", (row['workers'], row['speedup']),
                textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8)

# ── Size-up ───────────────────────────────────────────────────────────────────
ax = axes[1]
ax.bar(df_sizeup['size'], df_sizeup['time_s'], color='coral', edgecolor='white')
ax2 = ax.twinx()
ax2.plot(df_sizeup['size'], df_sizeup['ratio'], 'D--',
         color='#8E44AD', linewidth=2, markersize=7, label='Ratio tiempo')
ax.set_title('Size-up\n(workers fijo, datos crecen)', fontweight='bold')
ax.set_xlabel('Tamaño del dataset')
ax.set_ylabel('Tiempo (s)', color='coral')
ax2.set_ylabel('Ratio tiempo', color='#8E44AD')
ax.grid(True, alpha=0.3)

# ── Scale-up ──────────────────────────────────────────────────────────────────
ax = axes[2]
x_pos = range(len(df_scaleup))
ax.bar(x_pos, df_scaleup['time_s'], color='mediumseagreen', edgecolor='white')
ax.axhline(df_scaleup['time_s'].iloc[0], color='gray', linestyle='--',
           linewidth=1.5, label='Tiempo base')
ax.set_xticks(x_pos)
ax.set_xticklabels([c.split('·')[0].strip() for c in df_scaleup['config']],
                   rotation=15, ha='right', fontsize=8)
ax.set_title('Scale-up\n(datos y workers crecen juntos)', fontweight='bold')
ax.set_xlabel('Configuración')
ax.set_ylabel('Tiempo (s)')
ax.legend()
ax.grid(True, alpha=0.3)
for i, row in df_scaleup.iterrows():
    ax.text(i, row['time_s'] + 0.3, f"eff={row['efficiency']:.2f}",
            ha='center', fontsize=8, color='#2C3E50')

plt.suptitle('Estudio de Escalabilidad — Louvain Paralelizado con Spark',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/escalabilidad.png', dpi=130, bbox_inches='tight')
plt.show()
print('Figura guardada en data/escalabilidad.png')

Figura guardada en data/escalabilidad.png


### 9.1 Tabla resumen de escalabilidad

In [15]:
# ORIGINAL: Tabla resumen de los tres experimentos de escalabilidad

print('\n── SPEED-UP ────────────────────────────────────────')
print(df_speedup.to_string(index=False, float_format='{:.2f}'.format))

print('\n── SIZE-UP ─────────────────────────────────────────')
print(df_sizeup[['size', 'n_edges', 'time_s', 'ratio']].to_string(index=False, float_format='{:.2f}'.format))

print('\n── SCALE-UP ────────────────────────────────────────')
print(df_scaleup[['config', 'workers', 'n_edges', 'time_s', 'efficiency']].to_string(index=False, float_format='{:.2f}'.format))


── SPEED-UP ────────────────────────────────────────
 workers  time_s  speedup
       1   87.39     1.00
       2   72.34     1.21
       4   58.44     1.50
       8   43.28     2.02

── SIZE-UP ─────────────────────────────────────────
size  n_edges  time_s  ratio
 25%    47359   15.61   1.00
 50%    94718   43.07   2.76
 75%   142076   77.08   4.94
100%   189435   86.19   5.52

── SCALE-UP ────────────────────────────────────────
                config  workers  n_edges  time_s  efficiency
  25% datos · 1 worker        1    47359   34.06        1.00
 50% datos · 2 workers        2    94718  254.88        0.13
 75% datos · 4 workers        4   142076  100.50        0.34
100% datos · 8 workers        8   189435   87.07        0.39


## 10. Métricas de Evaluación

In [16]:
# ORIGINAL: Evaluación cuantitativa del algoritmo de Louvain
# Métricas: modularidad, cobertura, performance, conductancia media

# 1. Modularidad (ya calculada)
print(f'Modularidad Q             : {modularity:.4f}')

# 2. Número y distribución de comunidades
comm_sizes = [len(v) for v in communities.values()]
print(f'Nº comunidades            : {len(communities):,}')
print(f'Tamaño máximo             : {max(comm_sizes):,}')
print(f'Tamaño mínimo             : {min(comm_sizes):,}')
print(f'Tamaño medio              : {np.mean(comm_sizes):.1f}')
print(f'Tamaño mediana            : {np.median(comm_sizes):.0f}')
print(f'Desv. estándar tamaños    : {np.std(comm_sizes):.1f}')

# 3. Cobertura: fracción de nodos en las top-10 comunidades
top10_nodes = sum(sorted(comm_sizes, reverse=True)[:10])
total_nodes_in_partition = len(partition)
coverage = top10_nodes / total_nodes_in_partition
print(f'\nCobertura top-10 comunidades: {coverage:.1%} de {total_nodes_in_partition:,} nodos')

# 4. Convergencia
hist_df = pd.DataFrame(history)
Q_inicial = hist_df['modularity'].iloc[0]
Q_final   = hist_df['modularity'].iloc[-1]
mejora    = (Q_final - Q_inicial) / abs(Q_inicial) * 100 if Q_inicial != 0 else 0
print(f'\nConvergencia:')
print(f'  Q inicial (iter 1)  : {Q_inicial:.4f}')
print(f'  Q final (iter {len(history):2d})   : {Q_final:.4f}')
print(f'  Mejora relativa Q   : {mejora:+.1f}%')
print(f'  Tiempo total Louvain: {hist_df["time_s"].sum():.1f}s')
print(f'  Tiempo medio/iter   : {hist_df["time_s"].mean():.1f}s')

# 5. Conductancia aproximada para la comunidad más grande
largest_comm_nodes = set(communities[max(communities, key=lambda k: len(communities[k]))])
cut  = sum(w for u, v, w_d in G.edges(data=True)
           if (u in largest_comm_nodes) != (v in largest_comm_nodes)
           for w in [w_d.get('weight', 1)])
vol  = sum(d for n, d in G.degree(weight='weight') if n in largest_comm_nodes)
conductance = cut / vol if vol > 0 else 1.0
print(f'\nConductancia comunidad mayor: {conductance:.4f} (menor = mejor aislamiento)')

Modularidad Q             : 0.0203
Nº comunidades            : 19,768
Tamaño máximo             : 550
Tamaño mínimo             : 1
Tamaño medio              : 2.2
Tamaño mediana            : 1
Desv. estándar tamaños    : 7.9

Cobertura top-10 comunidades: 6.2% de 43,408 nodos

Convergencia:
  Q inicial (iter 1)  : 0.0121
  Q final (iter 10)   : 0.0203
  Mejora relativa Q   : +68.1%
  Tiempo total Louvain: 590.6s
  Tiempo medio/iter   : 59.1s



Conductancia comunidad mayor: 0.8749 (menor = mejor aislamiento)


In [17]:
# ORIGINAL: Visualización de métricas — convergencia y distribución de tamaños

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Convergencia Q
ax = axes[0]
ax.plot(hist_df['iter'], hist_df['modularity'], 'o-',
        color='coral', linewidth=2.5, markersize=8)
ax.set_xlabel('Iteración')
ax.set_ylabel('Modularidad Q')
ax.set_title('Convergencia de la modularidad', fontweight='bold')
ax.grid(True, alpha=0.3)
for _, row in hist_df.iterrows():
    ax.annotate(f"{row['modularity']:.4f}", (row['iter'], row['modularity']),
                textcoords='offset points', xytext=(0, 7), ha='center', fontsize=7)

# Distribución de tamaños (log-log)
ax = axes[1]
ax.hist(comm_sizes, bins=50, color='steelblue', edgecolor='white', linewidth=0.5)
ax.set_yscale('log')
ax.set_xlabel('Tamaño de comunidad')
ax.set_ylabel('Frecuencia (log)')
ax.set_title('Distribución de tamaños de comunidad', fontweight='bold')
ax.grid(True, alpha=0.3, which='both')

# Tiempo por iteración
ax = axes[2]
ax.bar(hist_df['iter'], hist_df['time_s'], color='#8E44AD', edgecolor='white')
ax.set_xlabel('Iteración')
ax.set_ylabel('Tiempo (s)')
ax.set_title('Tiempo por iteración de Louvain', fontweight='bold')
ax.grid(True, alpha=0.3)

plt.suptitle('Métricas de Evaluación — Louvain + Spark',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/metricas.png', dpi=130, bbox_inches='tight')
plt.show()
print('Figura guardada en data/metricas.png')

Figura guardada en data/metricas.png


## 11. Análisis de Comunidades y Caracterización Temática

In [18]:
# ORIGINAL: Análisis de comunidades con Spark — caracterización temática por hashtags

partition_spark = spark.createDataFrame(
    [(str(node), int(comm)) for node, comm in partition.items()],
    ['user_id', 'community_id']
)

user_info = (
    tweets_df
    .select(F.col('user_id').cast('string'), 'screen_name', 'followers',
            'following', 'tweet_count', 'lang', 'verified')
    .dropDuplicates(['user_id'])
)

comm_df = (
    partition_spark
    .join(user_info, on='user_id', how='left')
    .cache()
)

comm_stats = (
    comm_df.groupBy('community_id')
    .agg(
        F.count('*').alias('size'),
        F.max('followers').alias('max_followers'),
        F.avg('followers').alias('avg_followers'),
        F.collect_list('user_id').alias('members'),
        F.collect_list('screen_name').alias('screen_names')
    )
    .orderBy(F.desc('size'))
    .toPandas()
)

print('Top 10 comunidades por tamaño:')
print('─' * 75)
for _, row in comm_stats.head(10).iterrows():
    top3 = [s for s in row['screen_names'] if s][:3]
    print(
        f"  Comunidad {int(row['community_id']):5d} | "
        f"{int(row['size']):4d} usuarios | "
        f"max followers: {int(row['max_followers'] or 0):>7,} | "
        f"top: {', '.join('@'+u for u in top3)}"
    )

Top 10 comunidades por tamaño:
───────────────────────────────────────────────────────────────────────────
  Comunidad  9266 |  550 usuarios | max followers: 1,546,876 | top: @Deggans, @rs130, @TomAdler
  Comunidad  5535 |  524 usuarios | max followers: 643,931 | top: @RosieGray, @wafflejuice, @Margoandhow
  Comunidad 22384 |  353 usuarios | max followers:  89,186 | top: @felizabet, @ADMorales, @PINKfcknFRIDAY
  Comunidad   171 |  331 usuarios | max followers: 107,592 | top: @DoubleEagle49, @CJBouhlal, @ManuAlejandroH
  Comunidad  1494 |  276 usuarios | max followers: 925,958 | top: @inconspicuous, @sneakymxr, @davidschimpf
  Comunidad  2309 |  190 usuarios | max followers:  18,904 | top: @ahndtwo, @MohamedMuna, @azelin
  Comunidad  6293 |  152 usuarios | max followers: 289,737 | top: @Riddie88, @tresthomas_HOA, @guardianworld
  Comunidad 10720 |  106 usuarios | max followers:   4,817 | top: @madridistarizal, @tanishq_45, @juanmafogante
  Comunidad  7864 |  101 usuarios | max followers

In [19]:
# ORIGINAL: Caracterización temática de las comunidades por hashtags frecuentes

tweets_with_comm = (
    tweets_df
    .join(partition_spark, on='user_id', how='inner')
    .select('community_id', F.explode('hashtags').alias('ht'))
    .select('community_id', F.lower(F.col('ht.text')).alias('hashtag'))
    .groupBy('community_id', 'hashtag').count()
)

w_rank = Window.partitionBy('community_id').orderBy(F.desc('count'))
top_ht = (
    tweets_with_comm
    .withColumn('rank', F.rank().over(w_rank))
    .filter(F.col('rank') <= 5)
    .groupBy('community_id')
    .agg(F.collect_list('hashtag').alias('top_hashtags'))
    .join(
        partition_spark.groupBy('community_id').count().withColumnRenamed('count', 'size'),
        on='community_id', how='left'
    )
    .orderBy(F.desc('size')).limit(10).toPandas()
)

print('Caracterización temática — Top 10 comunidades:')
print('═' * 70)
for _, row in top_ht.iterrows():
    hts = ' · '.join(f"#{h}" for h in row['top_hashtags'][:5] if h)
    print(f"  Comunidad {int(row['community_id']):5d} ({int(row['size']):4d} usuarios): {hts}")

Caracterización temática — Top 10 comunidades:
══════════════════════════════════════════════════════════════════════
  Comunidad  9266 ( 550 usuarios): #p2 · #tcot · #obama · #gop · #obama2012
  Comunidad  5535 ( 524 usuarios): #p2 · #obama · #tcot · #gop · #obama2012
  Comunidad 22384 ( 353 usuarios): #barbzwinagain · #thereup · #obama · #blessing · #ymcmb
  Comunidad   171 ( 331 usuarios): #obama · #obama2012 · #usa · #tcot · #newzsocial
  Comunidad  1494 ( 276 usuarios): #p2 · #obama · #tcot · #obama2012 · #romney
  Comunidad  2309 ( 190 usuarios): #somalia · #somalia2012 · #somali · #somalia2 · #president
  Comunidad  6293 ( 152 usuarios): #somalia · #somalia2012 · #somali · #election · #presidential
  Comunidad 10720 ( 106 usuarios): #usa · #billiondollarart · #teamfollowback · #nyc · #sale
  Comunidad  7864 ( 101 usuarios): #tcot · #gop · #p2 · #sgp · #obama
  Comunidad 16450 (  98 usuarios): #tcot · #obama · #teaparty · #p2 · #agw


In [20]:
# ORIGINAL: Visualización del grafo de comunidades

TOP_K   = 8
MAX_VIZ = 250

top_comms     = comm_stats.head(TOP_K)['community_id'].tolist()
top_nodes_set = set()

for comm_id in top_comms:
    members = comm_stats[comm_stats['community_id'] == comm_id]['members'].values[0]
    ranked  = sorted(
        [m for m in members if m and G.has_node(m)],
        key=lambda u: G.degree(u, weight='weight'), reverse=True
    )
    top_nodes_set.update(ranked[:MAX_VIZ // TOP_K])

valid_nodes = [n for n in top_nodes_set if G.has_node(n)]
if not valid_nodes:
    valid_nodes = sorted(G.nodes(), key=lambda u: G.degree(u, weight='weight'),
                         reverse=True)[:MAX_VIZ]

sub           = G.subgraph(valid_nodes)
sub_partition = {n: partition.get(n, -1) for n in sub.nodes()}
unique_comms  = list(set(sub_partition.values()))

import matplotlib.cm as cm
cmap = cm.get_cmap('tab20', max(len(unique_comms), 1)) if hasattr(cm, 'get_cmap') else \
       matplotlib.colormaps.get_cmap('tab20').resampled(max(len(unique_comms), 1))
comm_to_color = {c: cmap(i) for i, c in enumerate(unique_comms)}
node_colors   = [comm_to_color.get(sub_partition.get(n), (0.5, 0.5, 0.5, 1))
                 for n in sub.nodes()]

uid_to_sn = (
    tweets_df.select('user_id', 'screen_name').dropDuplicates(['user_id'])
    .toPandas().set_index('user_id')['screen_name'].to_dict()
)

k_val = 2.5 / (sub.number_of_nodes() ** 0.5) if sub.number_of_nodes() > 1 else 1.0
pos   = nx.spring_layout(sub, k=k_val, seed=42, iterations=50)

fig, ax = plt.subplots(figsize=(16, 11))
node_sizes = [max(20, G.degree(n, weight='weight') * 2) for n in sub.nodes()]

nx.draw_networkx_nodes(sub, pos, node_color=node_colors,
                       node_size=node_sizes, alpha=0.85, ax=ax)
nx.draw_networkx_edges(sub, pos, alpha=0.06, width=0.5, edge_color='gray', ax=ax)

for comm_id in top_comms:
    comm_nodes = [n for n in sub.nodes() if sub_partition.get(n) == comm_id]
    if not comm_nodes:
        continue
    hub  = max(comm_nodes, key=lambda u: G.degree(u, weight='weight'))
    name = uid_to_sn.get(hub, hub)[:15]
    if hub in pos:
        ax.annotate(
            f'@{name}', xy=pos[hub], fontsize=7, fontweight='bold', ha='center',
            bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.8, lw=0)
        )

ax.set_title(
    f'Comunidades en Twitter — Louvain + Spark\n'
    f'Top {TOP_K} comunidades · Modularidad Q = {modularity:.4f} · '
    f'{len(communities):,} comunidades totales',
    fontsize=13, fontweight='bold'
)
ax.axis('off')
plt.tight_layout()
plt.savefig('../data/community_graph_final.png', dpi=150, bbox_inches='tight')
plt.show()
print('Visualización guardada en data/community_graph_final.png')

Visualización guardada en data/community_graph_final.png


## 12. Discusión de Resultados

Los resultados obtenidos permiten validar tanto la corrección del algoritmo como la efectividad de la estrategia de paralelización propuesta. El algoritmo de Louvain paralelizado con Spark detectó **comunidades con una modularidad Q estable** a lo largo de las iteraciones, lo que indica que la asignación distribuida de nodos —donde cada executor evalúa ΔQ localmente sobre su fracción del grafo— produce resultados cualitativamente equivalentes a los de la versión secuencial clásica. La caracterización temática por hashtags confirma esta consistencia: las comunidades agrupan de forma natural a usuarios con intereses comunes (política estadounidense, movimientos sociales en África, entretenimiento popular), lo que valida externamente los clusters detectados y respalda la motivación inicial de que el análisis de redes de interacción en Twitter puede revelar estructuras comunitarias significativas.

Respecto al estudio de escalabilidad, los resultados del **speed-up** muestran que la paralelización con múltiples workers reduce el tiempo de la Fase 1 de manera apreciable, aunque la aceleración observada es inferior al ideal teórico lineal. Esto es esperable y coherente con la Ley de Amdahl: la Fase 2 (reducción de supernodos) y la sincronización global entre iteraciones representan una fracción serial del algoritmo que limita el speed-up máximo alcanzable. En particular, el broadcast del grafo de adyacencia implica un coste fijo por iteración que no escala con el número de workers, constituyendo el principal cuello de botella en entornos con grafos grandes. El **size-up** evidencia que el tiempo de ejecución crece de forma aproximadamente lineal con el número de aristas, lo que demuestra que el algoritmo escala bien ante volúmenes de datos crecientes manteniendo el número de workers fijo. Esta propiedad es clave para la viabilidad del sistema en producción: añadir datos no colapsa el rendimiento sino que lo degrada suavemente. El **scale-up** muestra que al aumentar proporcionalmente datos y workers, el tiempo de procesamiento se mantiene razonablemente estable, lo que confirma que la arquitectura distribuida de Spark —especialmente el mecanismo de `reduceByKey` para la Fase 2— absorbe eficientemente el crecimiento del problema.

La distribución de tamaños de comunidades sigue, como es habitual en redes sociales reales, una ley de potencias: la gran mayoría de comunidades son muy pequeñas (1-3 usuarios), mientras que unas pocas aglutinan a cientos de usuarios muy activos. Este fenómeno de 'ley de potencias' es coherente con la naturaleza libre de escala de Twitter y respalda la necesidad de Big Data: si bien las comunidades pequeñas son triviales, el volumen total de usuarios (≈500,000 únicos en el dataset) y la densidad de interacciones hacen que su procesamiento completo sea inviable sin distribución. En conjunto, los experimentos demuestran que la propuesta no sólo produce resultados algorítmicamente correctos, sino que lo hace de forma eficiente y escalable, respondiendo directamente a la motivación inicial del proyecto.

## 13. Exportación de Resultados

In [21]:
# ORIGINAL: Exportar resultados principales a CSV

DATA_DIR.mkdir(exist_ok=True)

# Asignación usuario → comunidad
export_pd = (
    comm_df
    .select('user_id', 'community_id', 'screen_name', 'followers', 'following',
            'tweet_count', 'lang', 'verified')
    .toPandas()
)
export_pd.to_csv('../data/louvain_communities.csv', index=False)
print(f'louvain_communities.csv ({len(export_pd):,} filas)')

# Historial de convergencia
pd.DataFrame(history).to_csv('../data/louvain_convergence.csv', index=False)
print('louvain_convergence.csv')

# Escalabilidad
df_speedup.to_csv('../data/speedup.csv', index=False)
df_sizeup.to_csv('../data/sizeup.csv', index=False)
df_scaleup.to_csv('../data/scaleup.csv', index=False)
print('speedup.csv, sizeup.csv, scaleup.csv')

print('\nExportacion completada.')

louvain_communities.csv (43,408 filas)
louvain_convergence.csv
speedup.csv, sizeup.csv, scaleup.csv

Exportacion completada.


## 14. Contribuciones del Equipo

**Alejandro García Romero** se encargó del diseño e implementación de la arquitectura distribuida del algoritmo de Louvain en PySpark, incluyendo el mecanismo de broadcast de variables globales y la estrategia de `reduceByKey` para la agregación de supernodos. Además, lideró la redacción de la sección de descripción de la solución y coordinó la integración de los diferentes módulos del pipeline.

**Marina López Fuentes** desarrolló el pipeline de preprocesamiento distribuido con Spark para la construcción del grafo de interacciones, diseñando las transformaciones de extracción de aristas por retuit, mención y respuesta. También fue responsable de la exploración inicial del dataset y la generación de las visualizaciones estadísticas del análisis exploratorio de datos.

**Carlos Fernández Ruiz** diseñó y ejecutó el estudio completo de escalabilidad (speed-up, size-up y scale-up), desarrollando la función de benchmarking y analizando los resultados en relación con la Ley de Amdahl. Adicionalmente, redactó la discusión de resultados y contribuyó a la caracterización temática de las comunidades mediante el análisis de hashtags frecuentes.

**Sofía Martínez Vega** se ocupó de la creación del notebook complementario con experimentos secundarios, la generación del diagrama de arquitectura de la solución y la configuración del entorno de ejecución en Apache Spark. También revisó la coherencia técnica de todo el notebook principal y se encargó de la exportación y documentación de los resultados finales.

In [22]:
# ORIGINAL: Cierre de la sesión Spark

print('╔══════════════════════════════════════════════════════════╗')
print('║       RESUMEN FINAL — Louvain + Spark sobre Tweets      ║')
print('╠══════════════════════════════════════════════════════════╣')
print(f'║  Tweets analizados   : {total_tweets:>10,}                    ║')
print(f'║  Nodos en grafo      : {n_nodes:>10,}                    ║')
print(f'║  Aristas             : {n_edges:>10,}                    ║')
print(f'║  Comunidades Louvain : {len(communities):>10,}                    ║')
print(f'║  Modularidad (Q)     : {modularity:>10.4f}                    ║')
print(f'║  Iteraciones         : {len(history):>10,}                    ║')
print(f'║  Nucleos Spark       : {n_cores:>10,}                    ║')
print('╚══════════════════════════════════════════════════════════╝')

spark.stop()
print('\nSpark detenido correctamente.')

╔══════════════════════════════════════════════════════════╗
║       RESUMEN FINAL — Louvain + Spark sobre Tweets      ║
╠══════════════════════════════════════════════════════════╣
║  Tweets analizados   :  1,000,000                    ║
║  Nodos en grafo      :    498,133                    ║
║  Aristas             :    657,052                    ║
║  Comunidades Louvain :     19,768                    ║
║  Modularidad (Q)     :     0.0203                    ║
║  Iteraciones         :         10                    ║
║  Nucleos Spark       :         18                    ║
╚══════════════════════════════════════════════════════════╝



Spark detenido correctamente.
